# Day 9 — Run All RAG Practices

Run from top to bottom. This combines Practices 00–05 into one session. It uses Groq for generation, MiniLM for local embeddings, ChromaDB for storage, and Gradio for the final app.

In [ ]:
from pathlib import Path
import os
import sys

work_dir = Path.cwd()
if not (work_dir / 'practice_00_groq_test.py').exists():
    work_dir = work_dir / 'day9'
if not (work_dir / 'practice_00_groq_test.py').exists():
    raise FileNotFoundError('Could not find the Day 9 practice files.')
os.chdir(work_dir)
print('Working directory:', Path.cwd())

## Setup
Uncomment the install line if packages are missing. Store the Groq key in `day9/.env` as `GROQ_API_KEY=...`; the notebook never prints the key.

In [ ]:
# %pip install groq python-dotenv pypdf langchain-community langchain-text-splitters langchain-huggingface langchain-chroma chromadb sentence-transformers gradio

In [ ]:
from dotenv import load_dotenv

load_dotenv()
if not os.getenv('GROQ_API_KEY'):
    raise ValueError('GROQ_API_KEY is missing. Add it to day9/.env and rerun this cell.')
print('GROQ_API_KEY is available.')

## Practice 00 — Test the Groq connection

In [ ]:
from groq import Groq

groq_client = Groq(api_key=os.environ['GROQ_API_KEY'])
response = groq_client.chat.completions.create(
    model='llama-3.1-8b-instant',
    messages=[{'role': 'user', 'content': 'Explain RAG in one sentence.'}],
    temperature=0.2,
)
print('Groq connection successful!')
print(response.choices[0].message.content)

## Practice 01 — Load and chunk a PDF
Change `PDF_PATH` to use another PDF in the Day 9 folder.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

PDF_PATH = 'sample.pdf'
pdf_path = Path(PDF_PATH)
if not pdf_path.exists():
    raise FileNotFoundError(f'PDF not found: {pdf_path.resolve()}')

pages = PyPDFLoader(str(pdf_path)).load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=120,
    separators=['\n\n', '\n', '. ', ' ', ''],
)
chunks = splitter.split_documents(pages)
print(f'Pages loaded: {len(pages)}')
print(f'Total chunks: {len(chunks)}')
if pages:
    print('\nFirst page preview:\n', pages[0].page_content[:400])
if chunks:
    sample_index = min(1, len(chunks) - 1)
    print('\nSample chunk metadata:', chunks[sample_index].metadata)
    print(chunks[sample_index].page_content[:300])

## Practice 02 — Embed and store in ChromaDB
The first run may download the MiniLM embedding model. The notebook reuses an existing index unless `REBUILD_INDEX` is set to `True`.

In [ ]:
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = Path('chroma_rag_demo')
COLLECTION_NAME = 'rag_demo'
REBUILD_INDEX = False

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)
test_vector = embeddings.embed_query('What is the deadline?')
print('Vector dimensions:', len(test_vector))

if REBUILD_INDEX and CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

if not CHROMA_DIR.exists():
    vector_store = Chroma.from_documents(
        documents=chunks, embedding=embeddings,
        persist_directory=str(CHROMA_DIR),
        collection_name=COLLECTION_NAME,
    )
    print('Created a new Chroma index.')
else:
    vector_store = Chroma(
        persist_directory=str(CHROMA_DIR),
        embedding_function=embeddings,
        collection_name=COLLECTION_NAME,
    )
    print('Reusing the existing Chroma index.')
print('Stored vectors:', vector_store._collection.count())

In [ ]:
query = 'What is the attendance policy?'  # Change this
results = vector_store.similarity_search(query, k=2)
for index, doc in enumerate(results, start=1):
    page = doc.metadata.get('page', '?')
    print(f'\nResult {index} (page {page}):')
    print(doc.page_content[:300])

## Practice 03 — Retrieve and generate
Edit `question` and rerun the final cell to ask more questions without entering a CLI loop.

In [ ]:
TOP_K = 4
SYSTEM_PROMPT = '''You are a document-based AI assistant.
Answer ONLY using the context provided below.
If the answer is not in the context, say exactly:
I could not find that information in the provided documents.
Always include the page number when referencing a fact.'''
retriever = vector_store.as_retriever(search_kwargs={'k': TOP_K})

def format_context(docs):
    parts = []
    for index, doc in enumerate(docs, start=1):
        page = doc.metadata.get('page', '?')
        source = doc.metadata.get('source', 'unknown')
        parts.append(f'[Source {index} | {source} | page {page}]\n{doc.page_content}')
    return '\n\n---\n\n'.join(parts)

def ask(question: str):
    docs = retriever.invoke(question)
    print(f'Retrieved {len(docs)} chunks:')
    for doc in docs:
        print(f"- page {doc.metadata.get('page', '?')}: {doc.page_content[:100]}...")
    prompt = f'''Question: {question}

Retrieved context:
{format_context(docs)}

Answer using ONLY the context above. Include page references.'''
    response = groq_client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        temperature=0.2,
    )
    answer = response.choices[0].message.content
    print('\nAnswer:\n', answer)
    return answer, docs

In [ ]:
question = 'What are the main points in this document?'  # Change this
answer, retrieved_docs = ask(question)

## Practice 04 — Inspect retrieval quality

In [ ]:
def display_page(doc):
    page = doc.metadata.get('page')
    return page + 1 if isinstance(page, int) else '?'

def show_results(store, question: str, k: int):
    print(f'\nQuestion: {question} | top-k = {k}')
    for index, doc in enumerate(store.similarity_search(question, k=k), start=1):
        preview = doc.page_content[:350].replace('\n', ' ')
        print(f'\nResult {index} | page {display_page(doc)}')
        print(preview)

inspection_question = 'What happens if I submit my assignment late?'  # Change this
for k in [1, 2, 4, 8]:
    show_results(vector_store, inspection_question, k)

## Practice 05 — Launch the Gradio application
The app runs as a child process so the notebook remains usable. Open the URL printed by the process, normally `http://127.0.0.1:7860`. Run the stop cell when finished.

In [ ]:
import subprocess
import time

if 'gradio_process' in globals() and gradio_process.poll() is None:
    print('Gradio is already running at http://127.0.0.1:7860')
else:
    gradio_process = subprocess.Popen(
        [sys.executable, 'practice_05_gradio_app.py'],
        env=os.environ.copy(),
    )
    time.sleep(3)
    if gradio_process.poll() is None:
        print('Gradio started. Open http://127.0.0.1:7860')
    else:
        print('Gradio exited early. Check the output above for errors.')

In [ ]:
# Stop the Gradio server when finished.
if 'gradio_process' in globals() and gradio_process.poll() is None:
    gradio_process.terminate()
    gradio_process.wait(timeout=10)
    print('Gradio stopped.')
else:
    print('Gradio is not running from this notebook.')